**生成式 AI 使用说明**：本作业中使用生成式 AI 工具时，适用与协作相同的课程政策。和其他协作者一样，每位学生都必须独立写出自己的解答，不能直接依赖交互输出；提交内容中还应注明协作的性质。使用生成式 AI 工具实质性完成作业内容不符合本作业的精神，也会违反 [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 将你的 Google Drive 挂载到 Colab VM。
from google.colab import drive
drive.mount('/content/drive')

# TODO：填写 Drive 中保存解压后
# 作业文件夹，例如 'cs231n/assignments/assignment2/'
FOLDERNAME = "cs231n/assignments/assignment2/"
assert FOLDERNAME is not None, "[!] Enter the foldername."

# 现在已经挂载 Drive，下面确保
# Colab VM 的 Python 解释器可以加载
# 其中的 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 将 COCO 数据集下载到你的 Drive
# 如果它还不存在。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_coco_dataset.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# 使用 RNN 做图像描述
在这个练习中，你将实现普通循环神经网络（vanilla Recurrent Neural Networks），并用它们训练一个能够为图像生成新描述的模型。

In [ ]:

# 设置单元。
import time, os, json
import numpy as np
import torch
import matplotlib.pyplot as plt

from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.rnn_layers_pytorch import *
from cs231n.captioning_solver_pytorch import CaptioningSolverPytorch
from cs231n.classifiers.rnn_pytorch import CaptioningRNN
from cs231n.coco_utils import load_coco_data, sample_coco_minibatch, decode_captions
from cs231n.image_utils import image_from_url

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # 设置图像的默认大小。
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

import sys
import types
import importlib

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

def rel_error(x, y):
    """ returns relative error """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

# COCO 数据集
在这个练习中，我们将使用 2014 版 [COCO dataset](https://cocodataset.org/)，这是图像描述任务的标准测试平台。该数据集包含 80,000 张训练图像和 40,000 张验证图像，每张图像都由 Amazon Mechanical Turk 上的工作人员标注了 5 条描述。

**图像特征。** 我们已经为你预处理数据并提取好特征。对所有图像，我们从在 ImageNet 上预训练的 VGG-16 网络的 fc7 层提取特征，并将这些特征存储在 `train2014_vgg16_fc7.h5` 和 `val2014_vgg16_fc7.h5` 文件中。为了减少处理时间和内存需求，我们使用 Principal Component Analysis（PCA）把特征维度从 4096 降到 512，这些特征存储在 `train2014_vgg16_fc7_pca.h5` 和 `val2014_vgg16_fc7_pca.h5` 文件中。原始图像接近 20GB，因此没有包含在下载中。由于所有图像都来自 Flickr，我们把训练图像和验证图像的 URL 分别存储在 `train2014_urls.txt` 和 `val2014_urls.txt` 中。这样你可以在可视化时按需下载图像。

**描述。** 直接处理字符串效率较低，因此我们会使用编码后的描述。每个词都会被分配一个整数 ID，这样就可以把一条描述表示为整数序列。整数 ID 和单词之间的映射位于 `coco2014_vocab.json` 文件中，你可以使用 `cs231n/coco_utils.py` 文件中的 `decode_captions` 函数，把整数 ID 的 NumPy 数组转换回字符串。

**特殊 token。** 我们向词表中加入了几个特殊 token，并已经为你处理好相关实现细节。我们在每条描述开头添加特殊 `<START>` token，在结尾添加 `<END>` token。罕见词会被替换为特殊 `<UNK>` token（表示 “unknown”）。此外，由于训练 minibatch 中的描述长度不同，我们会在短描述的 `<END>` token 后填充特殊 `<NULL>` token，使所有描述长度相同，并且不会为 `<NULL>` token 计算损失或梯度。

你可以使用 `cs231n/coco_utils.py` 文件中的 `load_coco_data` 函数加载所有 COCO 数据（描述、特征、URL 和词表）。运行下面的单元执行加载：

In [ ]:
# 从磁盘加载 COCO 数据到字典中。
# 本作业剩余部分将使用降维后的特征；
# 你也可以通过修改下面的 flag，自行尝试使用原始特征。
data = load_coco_data(pca_features=True)

# 打印数据字典中的所有键和值。
for k, v in data.items():
    if type(v) == np.ndarray:
        print(k, type(v), v.shape, v.dtype)
    else:
        print(k, type(v), len(v))

## 检查数据
开始处理数据集之前，查看一些示例总是好习惯。

你可以使用 `cs231n/coco_utils.py` 文件中的 `sample_coco_minibatch` 函数，从 `load_coco_data` 返回的数据结构中采样 minibatch。运行下面的代码，采样一个小的训练 minibatch，并显示图像及其描述。多运行几次并查看结果，有助于你理解数据集。

In [ ]:
# Sample a minibatch 并 show images 并 captions.
# If you get an error, URL just no longer exists, so don't worry!
# 你可以 re-sample as many times as you want.
batch_size = 3

captions, features, urls = sample_coco_minibatch(data, batch_size=batch_size)
for i, (caption, url) in enumerate(zip(captions, urls)):
    plt.imshow(image_from_url(url))
    plt.axis('off')
    caption_str = decode_captions(caption, data['idx_to_word'])
    plt.title(caption_str)
    plt.show()

# 循环神经网络
如课上所述，我们将使用 Recurrent Neural Network（RNN）语言模型做图像描述。`cs231n/rnn_layers_pytorch.py` 文件包含循环神经网络所需的不同层类型实现，`cs231n/classifiers/rnn_pytorch.py` 文件使用这些层实现图像描述模型。

我们首先会在 `cs231n/rnn_layers_pytorch.py` 中实现不同类型的 RNN 层。

_如果想更深入理解 RNN，可选阅读：https://www.deeplearningbook.org/contents/rnn.html#pf7_

# Vanilla RNN：单步前向传播
打开 `cs231n/rnn_layers_pytorch.py` 文件。该文件实现了循环神经网络常用不同层的前向传播。注意，由于这里使用 PyTorch，反向传播会由 PyTorch 的 autograd 处理。

首先实现 `rnn_step_forward` 函数，它实现 vanilla RNN 单个时间步的前向传播。完成后运行下面的代码检查你的实现。你应该看到 `1e-8` 数量级或更小的误差。

In [ ]:
N, D, H = 3, 10, 4

x = torch.from_numpy(np.linspace(-0.4, 0.7, num=N*D).reshape(N, D))
prev_h = torch.from_numpy(np.linspace(-0.2, 0.5, num=N*H).reshape(N, H))
Wx = torch.from_numpy(np.linspace(-0.1, 0.9, num=D*H).reshape(D, H))
Wh = torch.from_numpy(np.linspace(-0.3, 0.7, num=H*H).reshape(H, H))
b = torch.from_numpy(np.linspace(-0.2, 0.4, num=H))

next_h = rnn_step_forward(x, prev_h, Wx, Wh, b).numpy()
expected_next_h = np.asarray([
  [-0.58172089, -0.50182032, -0.41232771, -0.31410098],
  [ 0.66854692,  0.79562378,  0.87755553,  0.92795967],
  [ 0.97934501,  0.99144213,  0.99646691,  0.99854353]])

print('next_h error: ', rel_error(expected_next_h, next_h))

# Vanilla RNN：单步反向传播
由于我们用 PyTorch 实现了 `rnn_step_forward`，因此**不需要**实现 `rnn_step_backward`。我们可以用数值梯度检查器验证 PyTorch autograd 的反向传播。

不过，如果你想挑战自己，也可以尝试自己实现 `rnn_step_backward`。本作业不要求这一点。

In [ ]:
from cs231n.rnn_layers_pytorch import rnn_step_forward

# 创建测试输入。
np.random.seed(231)
N, D, H = 4, 5, 6
x = torch.from_numpy(np.random.randn(N, D))
h = torch.from_numpy(np.random.randn(N, H))
Wx = torch.from_numpy(np.random.randn(D, H))
Wh = torch.from_numpy(np.random.randn(H, H))
b = torch.from_numpy(np.random.randn(H))

# 启用梯度跟踪，并执行 RNN 前向传播。
for tensor in [x, h, Wx, Wh, b]:
  tensor.requires_grad_()
next_h = rnn_step_forward(x, h, Wx, Wh, b)

# 模拟随机上游梯度，并使用 PyTorch autograd 执行反向传播。
dnext_h = torch.from_numpy(np.random.randn(*next_h.shape))
next_h.backward(dnext_h)

# 将梯度收集到单独的 numpy 数组中。
dx = x.grad.detach().numpy()
dh = h.grad.detach().numpy()
dWx = Wx.grad.detach().numpy()
dWh = Wh.grad.detach().numpy()
db = b.grad.detach().numpy()
dnext_h = dnext_h.detach().numpy()

# 同时将测试输入转换为 numpy 数组。
x =  x.detach().numpy()
h =  h.detach().numpy()
Wx = Wx.detach().numpy()
Wh = Wh.detach().numpy()
b =  b.detach().numpy()

# 包装前向传播，使其支持 numpy 数组输入和输出。
# 使用 `torch.no_grad()` 显式关闭梯度跟踪。
def rnn_step_forward_numpy(x, h, Wx, Wh, b):
  with torch.no_grad():
    return rnn_step_forward(
        torch.from_numpy(x),
        torch.from_numpy(h),
        torch.from_numpy(Wx),
        torch.from_numpy(Wh),
        torch.from_numpy(b),
    ).numpy()

# 计算数值梯度并进行比较。
fx = lambda x: rnn_step_forward_numpy(x, h, Wx, Wh, b)
fh = lambda h: rnn_step_forward_numpy(x, h, Wx, Wh, b)
fWx = lambda Wx: rnn_step_forward_numpy(x, h, Wx, Wh, b)
fWh = lambda Wh: rnn_step_forward_numpy(x, h, Wx, Wh, b)
fb = lambda b: rnn_step_forward_numpy(x, h, Wx, Wh, b)

dx_num = eval_numerical_gradient_array(fx, x, dnext_h)
dh_num = eval_numerical_gradient_array(fh, h, dnext_h)
dWx_num = eval_numerical_gradient_array(fWx, Wx, dnext_h)
dWh_num = eval_numerical_gradient_array(fWh, Wh, dnext_h)
db_num = eval_numerical_gradient_array(fb, b, dnext_h)

# 你应该看到误差在 1e-9 或更小的量级。
print('dx error: ', rel_error(dx_num, dx))
print('dh error: ', rel_error(dh_num, dh))
print('dWx error: ', rel_error(dWx_num, dWx))
print('dWh error: ', rel_error(dWh_num, dWh))
print('db error: ', rel_error(db_num, db))

# Vanilla RNN：前向传播
现在你已经实现了 vanilla RNN 单个时间步的前向传播，接下来要用它实现能处理整个序列数据的 RNN。

在 `cs231n/rnn_layers_pytorch.py` 文件中实现 `rnn_forward` 函数。它应该使用你上面定义的 `rnn_step_forward` 函数。完成后运行下面的代码检查你的实现。你应该看到 `1e-7` 数量级或更小的误差。

In [ ]:
from cs231n.rnn_layers_pytorch import rnn_forward

N, T, D, H = 2, 3, 4, 5

x = torch.from_numpy(np.linspace(-0.1, 0.3, num=N*T*D).reshape(N, T, D))
h0 = torch.from_numpy(np.linspace(-0.3, 0.1, num=N*H).reshape(N, H))
Wx = torch.from_numpy(np.linspace(-0.2, 0.4, num=D*H).reshape(D, H))
Wh = torch.from_numpy(np.linspace(-0.4, 0.1, num=H*H).reshape(H, H))
b = torch.from_numpy(np.linspace(-0.7, 0.1, num=H))

h = rnn_forward(x, h0, Wx, Wh, b).numpy()
expected_h = np.asarray([
  [
    [-0.42070749, -0.27279261, -0.11074945,  0.05740409,  0.22236251],
    [-0.39525808, -0.22554661, -0.0409454,   0.14649412,  0.32397316],
    [-0.42305111, -0.24223728, -0.04287027,  0.15997045,  0.35014525],
  ],
  [
    [-0.55857474, -0.39065825, -0.19198182,  0.02378408,  0.23735671],
    [-0.27150199, -0.07088804,  0.13562939,  0.33099728,  0.50158768],
    [-0.51014825, -0.30524429, -0.06755202,  0.17806392,  0.40333043]]])
print('h error: ', rel_error(expected_h, h))

# Vanilla RNN：反向传播
和前面一样，我们可以使用数值梯度检查器验证 PyTorch autograd 的反向传播。如果你愿意，也可以自己尝试实现 `rnn_step_backward`，但本作业不要求。

In [ ]:
from cs231n.rnn_layers_pytorch import rnn_forward

# 创建测试输入。
np.random.seed(231)
N, D, T, H = 2, 3, 10, 5
x = torch.from_numpy(np.random.randn(N, T, D))
h0 = torch.from_numpy(np.random.randn(N, H))
Wx = torch.from_numpy(np.random.randn(D, H))
Wh = torch.from_numpy(np.random.randn(H, H))
b = torch.from_numpy(np.random.randn(H))

# 启用梯度跟踪，并执行前向传播。
for tensor in [x, h0, Wx, Wh, b]:
  tensor.requires_grad_()
h = rnn_forward(x, h0, Wx, Wh, b)

# 模拟随机上游梯度，并使用 PyTorch autograd 执行反向传播。
dh = torch.from_numpy(np.random.randn(*h.shape))
h.backward(dh)

# 将梯度收集到单独的 numpy 数组中。
dx = x.grad.detach().numpy()
dh0 = h0.grad.detach().numpy()
dWx = Wx.grad.detach().numpy()
dWh = Wh.grad.detach().numpy()
db = b.grad.detach().numpy()
dh = dh.detach().numpy()

# 同时将测试输入转换为 numpy 数组。
x =  x.detach().numpy()
h0 =  h0.detach().numpy()
Wx = Wx.detach().numpy()
Wh = Wh.detach().numpy()
b =  b.detach().numpy()

# 包装前向传播，使其支持 numpy 数组输入和输出。
# 使用 `torch.no_grad()` 显式关闭梯度跟踪。
def rnn_forward_numpy(x, h0, Wx, Wh, b):
  with torch.no_grad():
    return rnn_forward(
        torch.from_numpy(x),
        torch.from_numpy(h0),
        torch.from_numpy(Wx),
        torch.from_numpy(Wh),
        torch.from_numpy(b),
    ).numpy()


fx = lambda x: rnn_forward_numpy(x, h0, Wx, Wh, b)
fh0 = lambda h0: rnn_forward_numpy(x, h0, Wx, Wh, b)
fWx = lambda Wx: rnn_forward_numpy(x, h0, Wx, Wh, b)
fWh = lambda Wh: rnn_forward_numpy(x, h0, Wx, Wh, b)
fb = lambda b: rnn_forward_numpy(x, h0, Wx, Wh, b)

dx_num = eval_numerical_gradient_array(fx, x, dh)
dh0_num = eval_numerical_gradient_array(fh0, h0, dh)
dWx_num = eval_numerical_gradient_array(fWx, Wx, dh)
dWh_num = eval_numerical_gradient_array(fWh, Wh, dh)
db_num = eval_numerical_gradient_array(fb, b, dh)

# 你应该看到误差在 1e-6 或更小的量级。
print('dx error: ', rel_error(dx_num, dx))
print('dh0 error: ', rel_error(dh0_num, dh0))
print('dWx error: ', rel_error(dWx_num, dWx))
print('dWh error: ', rel_error(dWh_num, dWh))
print('db error: ', rel_error(db_num, db))

# Word Embedding：前向传播
在深度学习系统中，我们通常用向量表示单词。词表中的每个单词都会关联一个向量，这些向量会和系统其余部分一起学习。

在 `cs231n/rnn_layers_pytorch.py` 文件中实现 `word_embedding_forward` 函数，把用整数表示的单词转换为向量。运行下面的代码检查你的实现。你应该看到 `1e-8` 数量级或更小的误差。

In [ ]:
N, T, V, D = 2, 4, 5, 3

x = torch.from_numpy(np.asarray([[0, 3, 1, 2], [2, 1, 0, 3]]))
W = torch.from_numpy(np.linspace(0, 1, num=V*D).reshape(V, D))

out = word_embedding_forward(x, W).numpy()
expected_out = np.asarray([
 [[ 0.,          0.07142857,  0.14285714],
  [ 0.64285714,  0.71428571,  0.78571429],
  [ 0.21428571,  0.28571429,  0.35714286],
  [ 0.42857143,  0.5,         0.57142857]],
 [[ 0.42857143,  0.5,         0.57142857],
  [ 0.21428571,  0.28571429,  0.35714286],
  [ 0.,          0.07142857,  0.14285714],
  [ 0.64285714,  0.71428571,  0.78571429]]])

print('out error: ', rel_error(expected_out, out))

# Word Embedding：反向传播
和前面一样，我们可以使用数值梯度检查器验证 PyTorch autograd 的反向传播。如果你愿意，也可以尝试自己实现 `word_embedding_backward`，但本作业不要求。

In [ ]:
np.random.seed(231)

N, T, V, D = 50, 3, 5, 6
x = torch.from_numpy(np.random.randint(V, size=(N, T)))
W = torch.from_numpy(np.random.randn(V, D))
W.requires_grad_()

out = word_embedding_forward(x, W)
dout = torch.from_numpy(np.random.randn(*out.shape))
out.backward(dout)

dW = W.grad.detach().numpy()
x = x.detach().numpy()
W = W.detach().numpy()
dout = dout.detach().numpy()

def word_embedding_forward_numpy(x, W):
  return word_embedding_forward(
      torch.from_numpy(x),
      torch.from_numpy(W),
  ).numpy()

f = lambda W: word_embedding_forward_numpy(x, W)
dW_num = eval_numerical_gradient_array(f, W, dout)

# 你应该看到误差在 1e-11 或更小的量级。
print('dW error: ', rel_error(dW, dW_num))

# Temporal Affine 层
在每个时间步，我们使用 affine 函数把该时间步的 RNN hidden vector 转换为词表中每个单词的分数。由于这和你在 Assignment 1 中实现的 affine 层非常类似，我们已经在 `temporal_affine_forward` 中为你提供了这个函数。运行下面的代码对实现做数值梯度检查。你应该看到 `1e-9` 数量级或更小的误差。

In [ ]:
np.random.seed(231)

# Gradient check 用于 temporal affine 层
N, T, D, M = 2, 3, 4, 5
x = torch.from_numpy(np.random.randn(N, T, D))
w = torch.from_numpy(np.random.randn(D, M))
b = torch.from_numpy(np.random.randn(M))

for tensor in [x, w, b]:
  tensor.requires_grad_()
out = temporal_affine_forward(x, w, b)
dout = torch.from_numpy(np.random.randn(*out.shape))
out.backward(dout)

dx = x.grad.detach().numpy()
dw = w.grad.detach().numpy()
db = b.grad.detach().numpy()

x = x.detach().numpy()
w = w.detach().numpy()
b = b.detach().numpy()
dout = dout.detach().numpy()

def temporal_affine_forward_numpy(x, w, b):
  return temporal_affine_forward(
      torch.from_numpy(x),
      torch.from_numpy(w),
      torch.from_numpy(b),
  ).numpy()

fx = lambda x: temporal_affine_forward_numpy(x, w, b)
fw = lambda w: temporal_affine_forward_numpy(x, w, b)
fb = lambda b: temporal_affine_forward_numpy(x, w, b)

dx_num = eval_numerical_gradient_array(fx, x, dout)
dw_num = eval_numerical_gradient_array(fw, w, dout)
db_num = eval_numerical_gradient_array(fb, b, dout)

print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))

# Temporal Softmax Loss
在 RNN 语言模型中，每个时间步都会为词表中的每个单词生成一个分数。我们知道每个时间步的真实单词，因此使用 softmax 损失函数计算该时间步的损失和梯度。我们把时间维度上的损失求和，并在 minibatch 上取平均。

但这里有一个细节：由于我们按 minibatch 操作，而不同描述长度可能不同，所以会在每条描述末尾追加 `<NULL>` token，使它们长度相同。我们不希望这些 `<NULL>` token 对损失或梯度有贡献，因此除了 scores 和真实标签外，损失函数还接收一个 `mask` 数组，用它说明 scores 中哪些元素应计入损失。

由于这和你在 Assignment 1 中实现的 softmax 损失函数非常类似，我们已经为你实现了这个损失函数；请查看 `cs231n/rnn_layers_pytorch.py` 文件中的 `temporal_softmax_loss` 函数。

运行下面的单元对损失做合理性检查，并对函数做数值梯度检查。你应该看到 `dx` 的误差在 `1e-7` 数量级或更小。

In [ ]:
# temporal softmax 损失的合理性检查。
from cs231n.rnn_layers_pytorch import temporal_softmax_loss

N, T, V = 100, 1, 10

def check_loss(N, T, V, p):
    x = 0.001 * torch.from_numpy(np.random.randn(N, T, V))
    y = torch.from_numpy(np.random.randint(V, size=(N, T)))
    mask = torch.from_numpy(np.random.rand(N, T)) <= p
    print(temporal_softmax_loss(x, y, mask).item())

check_loss(100, 1, 10, 1.0)   # 应接近 2.3
check_loss(100, 10, 10, 1.0)  # 应接近 23
check_loss(5000, 10, 10, 0.1) # 应位于 2.2 到 2.4 之间

# temporal softmax 损失的梯度检查。
np.random.seed(231231)
N, T, V = 7, 8, 9

x = torch.from_numpy(np.random.randn(N, T, V))
y = torch.from_numpy(np.random.randint(V, size=(N, T)))
mask = torch.from_numpy(np.random.rand(N, T) > 0.5)

x.requires_grad_()
loss = temporal_softmax_loss(x, y, mask, verbose=False)
loss.backward()
dx = x.grad.detach().numpy()
x = x.detach().numpy()
dx_num = eval_numerical_gradient(
    lambda x: temporal_softmax_loss(torch.from_numpy(x), y, mask), x, verbose=False)

print('dx error: ', rel_error(dx, dx_num))

# 用 RNN 做图像描述
现在你已经实现了必要的层，可以把它们组合起来构建图像描述模型。打开 `cs231n/classifiers/rnn_pytorch.py` 文件，查看 `CaptioningRNN` 类。

在 `loss` 函数中实现模型的前向传播。目前只需要实现 `cell_type='rnn'` 的 vanilla RNN 情况；LSTM 情况会稍后实现。完成后，运行下面的代码，用一个小测试用例检查你的前向传播；你应该看到 `1e-10` 数量级或更小的误差。

In [ ]:
N, D, W, H = 10, 20, 30, 40
word_to_idx = {'<NULL>': 0, 'cat': 2, 'dog': 3}
V = len(word_to_idx)
T = 13

model = CaptioningRNN(
    word_to_idx,
    input_dim=D,
    wordvec_dim=W,
    hidden_dim=H,
    cell_type='rnn',
    dtype=torch.float64
)

# Set 所有 模型 参数 到 fixed 值
for k, v in model.params.items():
    model.params[k] = torch.from_numpy(
        np.linspace(-1.4, 1.3, num=v.numel()).reshape(*v.shape))

features = torch.from_numpy(np.linspace(-1.5, 0.3, num=(N * D)).reshape(N, D))
captions = torch.from_numpy((np.arange(N * T) % V).reshape(N, T))

loss = model.loss(features, captions).item()
expected_loss = 9.83235591003

print('loss: ', loss)
print('expected loss: ', expected_loss)
print('difference: ', abs(loss - expected_loss))

运行下面的单元，对 `CaptioningRNN` 类做数值梯度检查；你应该看到 `1e-6` 数量级左右或更小的误差。

In [ ]:
np.random.seed(231)
torch.manual_seed(231)

batch_size = 2
timesteps = 3
input_dim = 4
wordvec_dim = 5
hidden_dim = 6
word_to_idx = {'<NULL>': 0, 'cat': 2, 'dog': 3}
vocab_size = len(word_to_idx)

captions = torch.from_numpy(np.random.randint(vocab_size, size=(batch_size, timesteps)))
features = torch.from_numpy(np.random.randn(batch_size, input_dim))

model = CaptioningRNN(
    word_to_idx,
    input_dim=input_dim,
    wordvec_dim=wordvec_dim,
    hidden_dim=hidden_dim,
    cell_type='rnn',
    dtype=torch.float64,
)

for k, v in model.params.items():
  v.requires_grad_()
loss = model.loss(features, captions)
loss.backward()
grads = {k: v.grad.detach().numpy() for k, v in model.params.items()}
for k, v in model.params.items():
  v.requires_grad_(False)

for param_name in sorted(grads.keys()):
    def fn(val):
      model.params[param_name] = torch.from_numpy(val)
      ret = model.loss(features, captions).numpy()
      return ret

    param_grad_num = eval_numerical_gradient(
        fn, model.params[param_name].numpy(), verbose=False, h=1e-6)

    e = rel_error(param_grad_num, grads[param_name])
    print('%s relative error: %e' % (param_name, e))

# 在小数据上过拟合 RNN 图像描述模型
类似于上一个作业中训练图像分类模型时使用的 `Solver` 类，本作业中我们使用 `CaptioningSolverPytorch` 类训练图像描述模型。打开 `cs231n/captioning_solver_pytorch.py` 文件，阅读 `CaptioningSolverPytorch` 类；它应该看起来很熟悉。

熟悉 API 后，运行下面的代码，确认你的模型能在 100 个训练样本的小样本上过拟合。最终损失应小于 0.1。

In [ ]:
np.random.seed(231)
torch.manual_seed(231)

small_data = load_coco_data(max_train=50)

small_rnn_model = CaptioningRNN(
    cell_type='rnn',
    word_to_idx=data['word_to_idx'],
    input_dim=data['train_features'].shape[1],
    hidden_dim=512,
    wordvec_dim=256,
)

small_rnn_solver = CaptioningSolverPytorch(
    small_rnn_model, small_data,
    num_epochs=50,
    batch_size=25,
    learning_rate=5e-3,
    verbose=True, print_every=10,
)

small_rnn_solver.train()

# Plot 训练 损失.
plt.plot(small_rnn_solver.loss_history)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training loss history')
plt.show()

打印最终训练损失。你应该看到最终损失小于 0.1。

In [ ]:
print('Final loss: ', small_rnn_solver.loss_history[-1])

# 测试时的 RNN 采样
和分类模型不同，图像描述模型在训练时和测试时的行为差别很大。训练时，我们能访问真实描述，因此每个时间步都把真实单词作为输入喂给 RNN。测试时，我们在每个时间步从词表分布中采样，并把采样得到的单词作为下一时间步的输入喂给 RNN。

在 `cs231n/classifiers/rnn_pytorch.py` 文件中实现 `sample` 方法，用于测试时采样。完成后，运行下面的代码，在训练数据和验证数据上用你的过拟合模型采样。训练数据上的样本应该非常好；验证数据上的样本则很可能不太合理。

In [ ]:
# If you get an error, URL just no longer exists, so don't worry!
# 你可以 re-sample as many times as you want.
for split in ['train', 'val']:
    minibatch = sample_coco_minibatch(small_data, split=split, batch_size=2)
    gt_captions, features, urls = minibatch
    gt_captions = decode_captions(gt_captions, data['idx_to_word'])

    sample_captions = small_rnn_model.sample(torch.from_numpy(features)).numpy()
    sample_captions = decode_captions(sample_captions, data['idx_to_word'])

    for gt_caption, sample_caption, url in zip(gt_captions, sample_captions, urls):
        img = image_from_url(url)
        # Skip missing URLs.
        if img is None: continue
        plt.imshow(img)
        plt.title('%s\n%s\nGT:%s' % (split, sample_caption, gt_caption))
        plt.axis('off')
        plt.show()

# 内联问题 1

在当前图像描述设置中，我们的 RNN 语言模型在每个时间步输出一个单词。不过，也可以把问题改写为让网络在_字符_级别上工作（例如 'a'、'b' 等），而不是在单词级别上工作：每个时间步接收前一个字符作为输入，并尝试预测序列中的下一个字符。例如，网络可能生成如下描述：

'A', ' ', 'c', 'a', 't', ' ', 'o', 'n', ' ', 'a', ' ', 'b', 'e', 'd'

你能描述一个使用字符级 RNN 的图像描述模型的优点吗？也请描述一个缺点。提示：有多个合理答案，但比较词级模型和字符级模型的参数空间可能会有帮助。

**你的回答：**